# W2C1 Lab: The N-gram Bard

Run every cell from the top. **Everything already works.**

Today you will:

1. Build a language model by counting word pairs.
2. Make it write sentences, and see why they wander.
3. Measure how surprised the model is, with perplexity.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import random
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

random.seed(0)

CORPUS = """the cat sat on the mat the cat ate the fish the dog sat on the rug
the dog chased the cat the cat ran up the tree the dog barked at the cat
the fish swam in the bowl the cat watched the fish the dog slept on the mat"""

words = CORPUS.split()
print(len(words), "words,", len(set(words)), "distinct")
print(words[:12])

## Part 1. A model is just counting

A bigram model answers one question: given the word I just saw, what
usually comes next? That is a dictionary of counts, nothing more.

In [ ]:
# GIVEN. Count what follows each word.
following = defaultdict(Counter)
for first, second in zip(words, words[1:]):
    following[first][second] += 1

print("after 'the' we have seen:")
for word, n in following["the"].most_common(6):
    print(f"   {word:<8} {n} times")

after_the = following["the"].most_common(6)
plt.figure(figsize=(6, 3))
plt.bar([w for w, n in after_the], [n for w, n in after_the], color="#7C2529")
plt.title("What follows 'the'?")
plt.ylabel("times seen")
plt.show()

In [ ]:
# GIVEN. Use those counts to write a sentence.
def generate(start="the", length=12):
    out = [start]
    for _ in range(length - 1):
        choices = following[out[-1]]
        if not choices:
            break
        options = list(choices.keys())
        weights = list(choices.values())
        out.append(random.choices(options, weights=weights)[0])
    return " ".join(out)

for i in range(4):
    print(generate())

In [ ]:
# ================== YOUR TURN 1 ==================
# Those sentences wander because the model only remembers ONE word.
# Build a TRIGRAM model instead: the key is the last TWO words.
#
# The counting code below is written for you. Just change N from 2 to 3
# and re-run to see the difference.
#
# Expected: the sentences become much more like the original text, because a
#           two-word memory pins down what comes next. Push N to 4 and it starts
#           reciting the corpus word for word: the model has memorised, not learned.
# ===============================================
N = 2                     # <-- change to 3, then 4

ngrams = defaultdict(Counter)
for i in range(len(words) - N + 1):
    key = tuple(words[i:i + N - 1])
    ngrams[key][words[i + N - 1]] += 1

def generate_n(length=12):
    out = list(words[:N - 1])
    for _ in range(length):
        choices = ngrams[tuple(out[-(N - 1):])]
        if not choices:
            break
        out.append(random.choices(list(choices), weights=list(choices.values()))[0])
    return " ".join(out)

print(f"N = {N}")
for i in range(4):
    print("  ", generate_n())

## Part 2. How surprised is the model?

Perplexity is the standard score for a language model. Low means the
model found the text predictable. It is the number every paper reports.

In [ ]:
# GIVEN. Perplexity of a sentence under the bigram model.
import math

def perplexity(sentence):
    toks = sentence.split()
    total_log_p = 0.0
    for first, second in zip(toks, toks[1:]):
        counts = following[first]
        # add-one smoothing so an unseen pair is unlikely, not impossible
        p = (counts[second] + 1) / (sum(counts.values()) + len(set(words)))
        total_log_p += math.log(p)
    return math.exp(-total_log_p / max(len(toks) - 1, 1))

for s in ["the cat sat on the mat",
          "the dog sat on the rug",
          "purple giraffes forecast quarterly earnings"]:
    print(f"{perplexity(s):8.1f}   {s}")

In [ ]:
# ================== YOUR TURN 2 ==================
# Write a sentence you think the model will find EASY (low perplexity)
# and one you think it will find HARD (high perplexity).
#
# Use only words from the corpus for the easy one.
#
# Expected: the easy sentence scores in the tens; the hard one scores in the
#           hundreds. A model is only ever confident about text that looks like
#           what it was trained on.
# ===============================================
EASY = "the cat sat on the mat"          # <-- your easy sentence
HARD = "purple giraffes forecast earnings"   # <-- your hard sentence

print(f"easy: {perplexity(EASY):8.1f}   {EASY}")
print(f"hard: {perplexity(HARD):8.1f}   {HARD}")
print()
print("easy really is lower:", perplexity(EASY) < perplexity(HARD))

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   N = 3 gives fluent, corpus-like sentences. N = 4 mostly recites the corpus.
#   That is the bias/variance trade in miniature: more context means better
#   local fluency and less ability to say anything new.
#
# YOUR TURN 2
#   Any in-corpus sentence scores low; any out-of-corpus wording scores high.
#   Add-one smoothing is what stops an unseen pair from giving probability 0,
#   which would make perplexity infinite.